# ML Prognostic Classifier using the 12-gene STAT3 Core Axis (TCGA - BRCA)

### Goal
Build and validate a Machine Learning Prognostic Classifier using the `ADAM10, C1GALT1, ESRRG, FZD6, GBE1, GLS, ITGA4, PDE3B, SGMS1, SLC11A2, SLC16A7, SLC22A1` signature across BRCA cohorts from TCGA.

### Purpose
To determine whether the multi-gene metabolic signature possesses independent non-linear prognostic utility for predicting 5-year overall survival, providing translational insight beyond traditional linear survival models.

### Interpretation
- **Cox Proportional Hazards C-index**: Establishes baseline linear survival predictive power.
- **Random Forest & MLP ROC-AUC**: Evaluates non-linear and interactive predictive power for 5-year binary overall survival.
- **Kaplan-Meier High vs Low Risk**: Stratifies patients based on machine learning risk scores to demonstrate clinical significance.

### Inputs/Parameters
- **Database:** `TCGA`
- **Cancers:** `BRCA`
- **Gene Signature:** `STAT3 Core Axis` (`ADAM10, C1GALT1, ESRRG, FZD6, GBE1, GLS, ITGA4, PDE3B, SGMS1, SLC11A2, SLC16A7, SLC22A1`)

### Outputs
- **Plots & Models:** Saved to `output/ml_prognostic_results/tcga/brca/12-gene/`


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# ML & Survival Modeling
from lifelines import CoxPHFitter, KaplanMeierFitter
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score, roc_curve

# If scikit-survival is installed, try importing RandomSurvivalForest
try:
    from sksurv.ensemble import RandomSurvivalForest
    SKSURV_AVAILABLE = True
except ImportError:
    SKSURV_AVAILABLE = False
    print("scikit-survival is not installed. Will fallback to standard Random Forest Classifier for binary 5-year OS status.")

import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 300


## 1. Data Loading

**Methodology:** We load the cleaned TCGA clinical and expression data that was pre-processed by our generator script. We also transpose and merge the expression data into the clinical cohort, yielding a unified pan-cancer dataset (if multiple cancers were specified).


In [ ]:
genes = ['ADAM10', 'C1GALT1', 'ESRRG', 'FZD6', 'GBE1', 'GLS', 'ITGA4', 'PDE3B', 'SGMS1', 'SLC11A2', 'SLC16A7', 'SLC22A1']
database = "tcga"
cancers = ['brca']

data_dir = '../input/tcga'
if not os.path.exists(data_dir):
    data_dir = 'input/tcga'

clin_dfs = []
expr_dfs = []

for cancer in cancers:
    clin_path = os.path.join(data_dir, f'{cancer}_clinical.csv')
    expr_path = os.path.join(data_dir, f'{cancer}_expression.csv')
    
    c_df = pd.read_csv(clin_path, low_memory=False)
    e_df = pd.read_csv(expr_path, low_memory=False)
    
    c_df['CANCER_TYPE'] = cancer.upper()
    
    # Preprocessing Expression for ML format
    e_df = e_df.drop(columns=['Entrez_Gene_Id'], errors='ignore')
    if 'Hugo_Symbol' in e_df.columns:
        e_df = e_df.set_index('Hugo_Symbol').T
    else:
        e_df = e_df.set_index(e_df.columns[0]).T
        
    e_df.index.name = 'SAMPLE_ID' if 'SAMPLE_ID' in c_df.columns else 'PATIENT_ID'
    e_df = e_df.reset_index()
    
    if database.lower() == 'tcga':
        # Align TCGA IDs
        e_df[e_df.columns[0]] = e_df[e_df.columns[0]].apply(lambda x: str(x)[:12])
        
    clin_dfs.append(c_df)
    expr_dfs.append(e_df)

# Pan-Cancer Merge
clin_df = pd.concat(clin_dfs, ignore_index=True)
expr_df = pd.concat(expr_dfs, ignore_index=True)

# Merge based on available ID type
merge_key = 'SAMPLE_ID' if 'SAMPLE_ID' in clin_df.columns else 'PATIENT_ID'
df = pd.merge(clin_df, expr_df, on=merge_key, how='inner')

# Drop duplicate patients if any
df = df.drop_duplicates(subset=['PATIENT_ID'])
print(f"Final dataset shape: {df.shape}")
if df.shape[0] == 0:
    raise ValueError("After merging clinical and expression data, the dataset is empty. Check if PATIENT_ID/SAMPLE_ID match between files.")
df.head()


In [ ]:
# Features & Target
present_genes = [g for g in genes if g in df.columns]
missing_genes = set(genes) - set(present_genes)
if len(present_genes) == 0:
    raise ValueError("None of the signature genes were found in the expression dataset.")
if len(missing_genes) > 0:
    print(f"Warning: The following genes were missing from the dataset and will be ignored: {missing_genes}")

num_genes = len(present_genes)
print(f"Proceeding with {num_genes} genes: {present_genes}")

# Output Directory
out_dir = f'../output/ml_prognostic_results/{database}/{"_".join(cancers)}/' + str(num_genes) + '-gene'
os.makedirs(out_dir, exist_ok=True)

df['5yr_survival'] = np.where((df['OS_MONTHS'] >= 60), 1, 
                              np.where((df['event'] == 1) & (df['OS_MONTHS'] < 60), 0, np.nan))

# train_test_split
X_train, X_test, y_train, y_test = train_test_split(df[present_genes], df, test_size=0.2, random_state=42)

print(f"Train size: {X_train.shape[0]}")
print(f"Test size: {X_test.shape[0]}")


## 2. Model 1: Baseline Cox Proportional Hazards Model


In [ ]:
cph_train = pd.concat([X_train, y_train[['OS_MONTHS', 'event']]], axis=1)
cph = CoxPHFitter(penalizer=0.1)
cph.fit(cph_train, duration_col='OS_MONTHS', event_col='event')

cph.print_summary()
c_index_cph = cph.concordance_index_
print(f"\nCox PH C-index on training set: {c_index_cph:.3f}")


In [ ]:
cph_test = pd.concat([X_test, y_test[['OS_MONTHS', 'event']]], axis=1)
c_index_test = cph.score(cph_test, scoring_method='concordance_index')
print(f"Cox PH C-index on TEST set: {c_index_test:.3f}")

plt.figure(figsize=(10, 6))
cph.plot()
plt.title(f'Cox PH Hazard Ratios ({num_genes}-gene Signature)')
plt.tight_layout()
plt.savefig(f"{out_dir}/cox_hazard_ratios.png")
plt.show()


## 3. Machine Learning Models (Random Forest & Neural Network)

**Methodology:**
This section evaluates the multi-gene signature's non-linear predictive power using two standard approaches:
1. **Random Forest Classifier**
2. **Multi-Layer Perceptron (Neural Network)**

Patients are labeled based on their **5-year Overall Survival (OS)** status (1 = survived > 5 years, 0 = died < 5 years). The models evaluate performance using the **ROC-AUC** metric, explicitly plot the overlapping ROC curves, and save the trained models for future inference.


In [ ]:
from sklearn.neural_network import MLPClassifier
import joblib

# Filter valid cases for binary classification (5-year survival)
train_bin = y_train.dropna(subset=['5yr_survival'])
test_bin = y_test.dropna(subset=['5yr_survival'])
X_train_bin = X_train.loc[train_bin.index]
X_test_bin = X_test.loc[test_bin.index]
y_train_bin = train_bin['5yr_survival']
y_test_bin = test_bin['5yr_survival']

# Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42, class_weight='balanced')
rf.fit(X_train_bin, y_train_bin)

# Neural Net (MLP)
mlp = MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=500, random_state=42)
mlp.fit(X_train_bin, y_train_bin)

joblib.dump(rf, f"{out_dir}/rf_model.pkl")
joblib.dump(mlp, f"{out_dir}/mlp_model.pkl")

# Generate Risk scores (we use 1 - prob(survival) as risk score)
rf_risk_test = 1 - rf.predict_proba(X_test_bin)[:, 1]
mlp_risk_test = 1 - mlp.predict_proba(X_test_bin)[:, 1]

# Plot ROC Curves
plt.figure(figsize=(8, 6))

fpr_rf, tpr_rf, _ = roc_curve(1 - y_test_bin, rf_risk_test) # predicting 'event' (death) within 5yr
auc_rf = roc_auc_score(1 - y_test_bin, rf_risk_test)

fpr_mlp, tpr_mlp, _ = roc_curve(1 - y_test_bin, mlp_risk_test)
auc_mlp = roc_auc_score(1 - y_test_bin, mlp_risk_test)

plt.plot(fpr_rf, tpr_rf, color='darkorange', lw=2, label=f'Random Forest (AUC = {auc_rf:.3f})')
plt.plot(fpr_mlp, tpr_mlp, color='green', lw=2, label=f'MLP Neural Net (AUC = {auc_mlp:.3f})')
plt.plot([0,1], [0,1], 'k--', lw=2, label='Random Chance')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title(f'ROC Curve: 5-Year Survival Prediction ({num_genes}-gene Signature)')
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig(f"{out_dir}/roc_curves.png")
plt.show()

# Plot Feature Importances from Random Forest
feature_imp_rf = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
plt.figure(figsize=(10, 6))
sns.barplot(x=feature_imp_rf.values, y=feature_imp_rf.index, palette='viridis')
plt.title(f'RF Gini Importances ({num_genes}-gene Signature)')
plt.tight_layout()
plt.savefig(f"{out_dir}/rf_feature_importances.png")
plt.show()


## 4. Kaplan-Meier Survival Analysis (High vs Low Risk)


In [ ]:
cph_risk_test = cph.predict_partial_hazard(X_test)
median_risk = np.median(cph_risk_test)
high_risk_mask = cph_risk_test > median_risk

test_high = y_test[high_risk_mask]
test_low = y_test[~high_risk_mask]

kmf = KaplanMeierFitter()
plt.figure(figsize=(8, 6))

kmf.fit(test_low['OS_MONTHS'], event_observed=test_low['event'], label=f'Low Risk (n={len(test_low)})')
ax = kmf.plot()

kmf.fit(test_high['OS_MONTHS'], event_observed=test_high['event'], label=f'High Risk (n={len(test_high)})')
kmf.plot(ax=ax)

try:
    from lifelines.statistics import multivariate_logrank_test
    res = multivariate_logrank_test(y_test['OS_MONTHS'], high_risk_mask, y_test['event'])
    plt.text(0.05, 0.05, f"Log-rank p-value: {res.p_value:.2e}", transform=ax.transAxes, fontsize=12)
except Exception as e:
    pass

plt.title(f'Kaplan-Meier Survival Curve: High vs Low Risk ({num_genes}-gene Signature)')
plt.xlabel('Time (Months)')
plt.ylabel('Survival Probability')
plt.tight_layout()
plt.savefig(f"{out_dir}/km_survival_curve.png")
plt.show()


In [ ]:
import subprocess
import sys
import os

try:
    from pan_cancer_config import ANALYSIS_SUFFIX
except ImportError:
    ANALYSIS_SUFFIX = ''

notebook_filename = 'ml_prognostic_classifier.ipynb'
output_base = 'ml_prognostic_classifier_report' + ANALYSIS_SUFFIX

jupyter_bin = os.path.join(os.path.dirname(sys.executable), 'jupyter')
if not os.path.exists(jupyter_bin): jupyter_bin = 'jupyter'

cmd_html = [jupyter_bin, "nbconvert", "--to", "html", notebook_filename, "--output-dir", out_dir, "--output", output_base]
res_html = subprocess.run(cmd_html, capture_output=True, text=True)

if res_html.returncode == 0:
    print(f"🎉 SUCCESS: Notebook successfully exported to '{os.path.join(out_dir, output_base)}.html'")
else:
    print("❌ HTML export failed.")
    print(res_html.stderr)
